In [1]:
import os
import re
from dotenv import load_dotenv
from langchain_core.documents import Document
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from uuid import uuid4
from langchain_groq import ChatGroq

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

from langchain_core.runnables import RunnableLambda

d:\AI-ML\AI Projects\RAG Chrome Extention\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
groq_api = os.getenv('GROQ_API_KEY')

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.7
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8936.42it/s]


In [3]:
llm.invoke('Hi')

AIMessage(content='How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 36, 'total_tokens': 44, 'completion_time': 0.027857184, 'completion_tokens_details': None, 'prompt_time': 0.003239786, 'prompt_tokens_details': None, 'queue_time': 0.05206747, 'total_time': 0.03109697}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019fad57-e4e9-7883-9865-f7694ef0b7c7-0', usage_metadata={'input_tokens': 36, 'output_tokens': 8, 'total_tokens': 44})

In [34]:
def load_youtube_snippets(url: str, window_seconds: int = 60, overlap_seconds: int = 10):

    match = re.search(r"(?:v=|youtu\.be/)([\w-]+)", url)

    if not match:
        raise ValueError(f"Could not extract video ID from URL: {url}")
    
    video_id = match.group(1)
    transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en', 'bn', 'hi'])
    snippets = transcript.snippets
    
    chunks = []
    current_text = []
    window_start = snippets[0].start if snippets else 0.0
    last_end = window_start

    for snippet in snippets:
        current_text.append(snippet.text)
        last_end = snippet.start + snippet.duration

        if last_end - window_start >= window_seconds:
            chunks.append(Document(
                page_content=" ".join(current_text).strip(),
                metadata={"source": video_id, "start": round(window_start, 2), "end": round(last_end, 2)}
            ))
            overlap_start = max(window_start, last_end - overlap_seconds)
            current_text = [s.text for s in snippets if overlap_start <= s.start < last_end]
            window_start = overlap_start

    if current_text:
        chunks.append(Document(
            page_content=" ".join(current_text).strip(),
            metadata={"source": video_id, "start": round(window_start, 2), "end": round(last_end, 2)}
        ))

    return chunks

texts = load_youtube_snippets("https://www.youtube.com/watch?v=tL9Lw250spc")
whole_content = ' '.join(doc.page_content for doc in texts)


In [35]:
pc = Pinecone()
index_name = "rag-extention"  # change if desired

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

vector_store = PineconeVectorStore(index=index, embedding=embedding_model)

In [36]:
uuids = [str(uuid4()) for _ in range(len(texts))]
vector_store.add_documents(documents=texts, ids=uuids)

['59ccbca2-0165-4e27-9573-9c4165d26b78',
 'f2d8f2c3-f07e-4616-ac3b-45993f6f7333',
 'e9576860-9d68-4037-93ff-714e9a8a0c40',
 '1126731d-5488-4ff1-ad39-1def086168e7',
 '5ed40017-6685-4b65-a4c6-aed9fddc23fd',
 'b86e2e24-066d-4c59-ac62-d25b8f556e0f',
 '3f788c7a-61b4-4d8b-9b9b-1c6aa1f62107',
 '2133af7b-7f8d-4a88-935f-cb8f287bdfcf',
 '57c51a3b-8e57-40f6-8af8-125fb7706131',
 '3440edda-dbe6-49f2-acb3-b02ad6c704f0',
 '82657805-810e-461f-b0b3-4fba2f9eac74',
 '35ac4f5e-ed64-45f8-a7b7-9c6da1440280',
 '1f0570fa-78f2-41c3-b15f-1403d92e5d14',
 'fc1a84d3-cc10-4982-bb42-d7891ac169c6',
 'daece7ed-d1d2-4e15-be00-a20bc26988d1',
 'f5bd7afc-e720-4a5e-9b3d-ea3bcf7a82c9',
 '2e9f2f88-1566-4f51-aafd-a463c44d09b7',
 'dc1a0a46-fa54-4ecf-8b4a-2066bf30962a',
 '6207c202-9905-424f-8332-43c30370a0fa',
 '1b712f7c-27a6-49cd-9c5d-8b52c5d0382d',
 '1b11fe8f-bbdf-4cd5-83e8-19269937d60e',
 '875f4f7c-477e-474d-90c6-5d30db68bd27',
 'ceef108a-6a39-40ed-9189-0dbe622fb9a1',
 'edc18d20-8a92-4c59-bdec-ef5ebd017106',
 '9e023d1a-3f04-

In [64]:
parser = StrOutputParser()

prompt_summarize = PromptTemplate(
        template="""You are a YouTube video summarizer. You will be given several transcript excerpts 
        retrieved from the same video (not multiple different videos). These excerpts may be out of order, 
        overlapping, or only cover parts of the video — treat them as fragments of one single, continuous video.

        Your task: answer the query below using ONLY the information in the excerpts. Synthesize the excerpts 
        into one coherent, unified response — do not treat them as separate items or list them one by one. 
        If the query asks for a summary, describe the video's overall narrative and key points in a natural, 
        flowing way, as if you had watched the whole video yourself.

        If the excerpts don't contain enough information to fully answer the query, say so honestly rather 
        than guessing or filling in gaps with outside knowledge.

        Query: {query}

        Transcript excerpts:
        {description}
    """,
    input_variables=['query', 'description']
    )

In [65]:
query =  "Summarize this video"

In [66]:
class SimilarQueries(BaseModel):
    queries: list[str] = Field(description="3 alternative phrasings of the given query")

prompt_query = PromptTemplate(
    template="""## Generate 3 alternative phrasings of the following query, 
    each capturing the same intent in different words.

    query: {query}
    """,
    input_variables=['query']
)

structured_llm = llm.with_structured_output(SimilarQueries)
chain = prompt_query | structured_llm
result = chain.invoke({'query': query})

all_queries = [query] + result.queries

In [ ]:
original_query = all_queries[0]
video_id = texts[0].metadata["source"]
seen = set()
deduped_chunks = []

for q in all_queries:
    results = vector_store.max_marginal_relevance_search(q, k=3, filter={"source": video_id})
    for doc in results:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            deduped_chunks.append(doc.page_content)

description_text = "\n\n".join(
    f"[Excerpt {i+1}]\n{chunk}" for i, chunk in enumerate(deduped_chunks)
)

chain = prompt_summarize | llm | parser



In [72]:
print(description_text)

[Excerpt 1]
and science learners in your life. It's the kind of tool I wish
I had when I was in school. So click the link below
or scan the QR code to get started with
Brilliant's tutor for free, or upgrade to premium
to unlock all courses. And right now, Veritasium
viewers can save 20% off an annual subscription at
brilliant.org/veritasium. So I want to thank Brilliant
for sponsoring this video, and I want to thank you for watching.

[Excerpt 2]
and it soon came to be known as WBE Theory, after their initials. - And the theory is quite rigid. It makes very specific predictions. This is good science. This is good science. Okay? This is sticking your neck out, and it's an incredibly beautiful theory. (mellow music)
- [Derek] Take a look at this table from Geoffrey West's book. These are the scaling
exponents WBE theory predicts, including many that are
not multiples of a quarter, but all follow from the same theory. For example, the radius
of an animal's aorta should scale with its mass

In [75]:
output = chain.invoke({
    'query': original_query,
    'description': whole_content
})

print(output)

The video discusses the concept of scaling laws in biology and how they apply to various aspects of life, from the metabolic rate of animals to the growth of cities. The narrative begins with the story of Tusko, an elephant that was given a large dose of LSD in the 1960s as part of a scientific experiment. The researchers had assumed that the safe dose for an elephant would be proportional to its mass, but this assumption proved to be incorrect, and Tusko died shortly after the experiment.

The video then explores the concept of scaling laws, which describe how different biological traits change as an organism's size increases. The researchers discuss how metabolic rate, heart rate, and lifespan all follow specific scaling laws, with metabolic rate being proportional to mass to the three-quarters power, according to Kleiber's Law. This law was first proposed by Max Kleiber in the 1930s and has been widely accepted as a fundamental principle of biology.

However, the video also notes th

In [76]:
output = chain.invoke({
    'query': original_query,
    'description': description_text
})

print(output)

This video explores the concept of scaling in biology, specifically how different characteristics of living organisms, such as metabolic rate, heart rate, and lifespan, change as their size increases. The hosts discuss how scientists West, Brown, and Enquist developed a theory, known as WBE Theory, which predicts how these characteristics scale with size. The theory is based on the idea that the networks that distribute resources within an organism, such as blood vessels and lungs, are space-filling and follow specific mathematical patterns.

The video highlights how WBE Theory makes specific predictions about the scaling of various biological traits, such as the radius of an animal's aorta and the area of its lungs, which are supported by empirical data. The hosts also discuss how the theory explains Kleiber's Law, which states that an organism's metabolic rate scales with its mass to the three-quarters power.

In addition to exploring the theoretical framework, the video touches on s

In [77]:
prompt_full_summary = PromptTemplate(
    template="""You are a YouTube video summarizer. Below is the full transcript of a video.
Write a comprehensive, well-organized summary covering the video's narrative arc from 
beginning to end — don't omit the opening story, examples, or any counterarguments presented.

Transcript:
{transcript}
""",
    input_variables=['transcript']
)

full_summary_chain = prompt_full_summary | llm | parser
full_summary = full_summary_chain.invoke({'transcript': whole_content})

In [78]:
print(full_summary)

The video begins with a thought-provoking question: how much LSD should be given to an elephant? The answer, of course, is none, but this leads to a discussion of a 1960s CIA experiment, MKUltra, which aimed to explore the effects of LSD on human behavior. As part of this project, researchers considered administering LSD to an elephant to study the potential release of an LSD-like substance in the elephant's brain, which might trigger a change in behavior.

To determine the dosage, scientists looked at the safe dose for cats, which was around 0.3 milligrams. Assuming a linear relationship between mass and dosage, they calculated that an elephant, being around 1,000 times larger than a cat, should receive 1,000 times the dose, approximately 300 milligrams of LSD. However, this calculation proved to be disastrous, as the elephant, Tusko, died shortly after receiving the dose.

The mistake made by the researchers was assuming that the safe dosage scales linearly with mass. This assumption

In [ ]:
"""
YouTube RAG pipeline — rebuilt as composable LangChain Runnables.

Design:
    A single dict flows through two LCEL chains:

        ingest_chain  : {url}          -> {url, video_id, texts, whole_content, already_indexed}
        answer_chain  : {..., query}   -> {..., intent, answer}

    full_pipeline = ingest_chain | answer_chain

Everything is a Runnable (RunnableLambda / RunnablePassthrough.assign / RunnableBranch),
so each stage can be swapped, batched, or reused independently — e.g. you can call
`ingest_chain.batch([...])` to pre-index many videos concurrently, or reuse
`answer_chain` alone once a video is already indexed.
"""

import os
import re
from typing import Literal

from dotenv import load_dotenv
from uuid import uuid4

from youtube_transcript_api import YouTubeTranscriptApi

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
    RunnableBranch,
)

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

from pydantic import BaseModel, Field

# --------------------------------------------------------------------------
# 1. Setup — models, vector store
# --------------------------------------------------------------------------

load_dotenv()

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.3)  # lower temp for grounded answers
llm_creative = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.7)  # for query paraphrasing

pc = Pinecone()
INDEX_NAME = "rag-extention"

if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(INDEX_NAME)
vector_store = PineconeVectorStore(index=index, embedding=embedding_model)

parser = StrOutputParser()

WINDOW_SECONDS = 60
OVERLAP_SECONDS = 10


# --------------------------------------------------------------------------
# 2. Ingestion — plain functions, wrapped as Runnables below
# --------------------------------------------------------------------------

def _extract_video_id(inputs: dict) -> str:
    match = re.search(r"(?:v=|youtu\.be/)([\w-]+)", inputs["url"])
    if not match:
        raise ValueError(f"Could not extract video ID from URL: {inputs['url']}")
    return match.group(1)


def _fetch_and_chunk(video_id: str) -> list[Document]:
    transcript = YouTubeTranscriptApi().fetch(video_id, languages=["en", "bn", "hi"])
    snippets = transcript.snippets

    chunks: list[Document] = []
    current_text: list[str] = []
    window_start = snippets[0].start if snippets else 0.0
    last_end = window_start

    for snippet in snippets:
        current_text.append(snippet.text)
        last_end = snippet.start + snippet.duration

        if last_end - window_start >= WINDOW_SECONDS:
            chunks.append(Document(
                page_content=" ".join(current_text).strip(),
                metadata={"source": video_id, "start": round(window_start, 2), "end": round(last_end, 2)},
            ))
            overlap_start = max(window_start, last_end - OVERLAP_SECONDS)
            current_text = [s.text for s in snippets if overlap_start <= s.start < last_end]
            window_start = overlap_start

    if current_text:
        chunks.append(Document(
            page_content=" ".join(current_text).strip(),
            metadata={"source": video_id, "start": round(window_start, 2), "end": round(last_end, 2)},
        ))

    return chunks


def _is_already_indexed(video_id: str) -> bool:
    """Cheap existence check via a metadata-filtered dummy query."""
    probe = index.query(
        vector=[0.0] * 384,
        filter={"source": video_id},
        top_k=1,
        include_metadata=False,
    )
    return len(probe.get("matches", [])) > 0


def _load_video(inputs: dict) -> dict:
    """Always fetch + chunk (cheap); only upsert to Pinecone if not already indexed."""
    video_id = inputs["video_id"]
    texts = _fetch_and_chunk(video_id)
    whole_content = " ".join(doc.page_content for doc in texts)

    already_indexed = _is_already_indexed(video_id)
    if not already_indexed:
        uuids = [str(uuid4()) for _ in texts]
        vector_store.add_documents(documents=texts, ids=uuids)

    return {**inputs, "texts": texts, "whole_content": whole_content, "already_indexed": already_indexed}


# ingest_chain: {url} -> {url, video_id, texts, whole_content, already_indexed}
ingest_chain = (
    RunnablePassthrough.assign(video_id=RunnableLambda(_extract_video_id))
    | RunnableLambda(_load_video)
)


# --------------------------------------------------------------------------
# 3. Intent classification — route "summarize the whole thing" vs "answer this"
# --------------------------------------------------------------------------

class QueryIntent(BaseModel):
    intent: Literal["full_summary", "specific_question"] = Field(
        description=(
            "'full_summary' if the user wants an overview/summary of the whole video. "
            "'specific_question' if they're asking about a particular detail, fact, or topic."
        )
    )


prompt_intent = PromptTemplate(
    template="""Classify the following user query about a video.

query: {query}
""",
    input_variables=["query"],
)

classify_intent = prompt_intent | llm.with_structured_output(QueryIntent)


# --------------------------------------------------------------------------
# 4. Multi-query expansion (for the retrieval path)
# --------------------------------------------------------------------------

class SimilarQueries(BaseModel):
    queries: list[str] = Field(description="3 alternative phrasings of the given query")


prompt_query_expand = PromptTemplate(
    template="""Generate 3 alternative phrasings of the following query,
each capturing the same intent in different words.

query: {query}
""",
    input_variables=["query"],
)

expand_query = prompt_query_expand | llm_creative.with_structured_output(SimilarQueries)


def _build_all_queries(inputs: dict) -> list[str]:
    result = expand_query.invoke({"query": inputs["query"]})
    return [inputs["query"]] + result.queries


# --------------------------------------------------------------------------
# 5. Retrieval — batched across all query variants, deduped, MMR + metadata filter
# --------------------------------------------------------------------------

def _single_query_search(inputs: dict) -> list[Document]:
    return vector_store.max_marginal_relevance_search(
        inputs["query"],
        k=3,
        fetch_k=10,
        filter={"source": inputs["video_id"]},
    )


single_query_retriever = RunnableLambda(_single_query_search)


def _retrieve_and_dedupe(inputs: dict) -> str:
    all_queries = _build_all_queries(inputs)
    video_id = inputs["video_id"]

    # .batch() runs these concurrently (thread pool) instead of a sequential for-loop —
    # this is the scalability win over the original notebook's serial loop.
    batched_inputs = [{"query": q, "video_id": video_id} for q in all_queries]
    results_per_query = single_query_retriever.batch(batched_inputs)

    seen = set()
    deduped_chunks = []
    for results in results_per_query:
        for doc in results:
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                deduped_chunks.append(doc.page_content)

    return "\n\n".join(
        f"[Excerpt {i + 1}]\n{chunk}" for i, chunk in enumerate(deduped_chunks)
    )


# --------------------------------------------------------------------------
# 6. Answer prompts
# --------------------------------------------------------------------------

prompt_retrieval_answer = PromptTemplate(
    template="""You are a YouTube video summarizer. You will be given several transcript excerpts
        retrieved from the same video (not multiple different videos). These excerpts may be out of order,
        overlapping, or only cover parts of the video — treat them as fragments of one single, continuous video.

        Your task: answer the query below using ONLY the information in the excerpts. Synthesize the excerpts
        into one coherent, unified response — do not treat them as separate items or list them one by one.

        If the excerpts don't contain enough information to fully answer the query, say so honestly rather
        than guessing or filling in gaps with outside knowledge.

        Query: {query}

        Transcript excerpts:
        {description}
    """,
    input_variables=["query", "description"],
)

prompt_full_summary = PromptTemplate(
    template="""You are a YouTube video summarizer. Below is the full transcript of a video.
        Write a comprehensive, well-organized summary covering the video's narrative arc from
        beginning to end — don't omit the opening story, examples, or any counterarguments presented.\n
        Transcript:
        {transcript}
    """,
    input_variables=["transcript"],
)

retrieval_answer_chain = (
    RunnablePassthrough.assign(description=RunnableLambda(_retrieve_and_dedupe))
    | prompt_retrieval_answer
    | llm
    | parser
)

full_summary_chain = (
    RunnableLambda(lambda inputs: {"transcript": inputs["whole_content"]})
    | prompt_full_summary
    | llm
    | parser
)


# --------------------------------------------------------------------------
# 7. Route by intent, then compose the full pipeline
# --------------------------------------------------------------------------

answer_chain = (
    RunnablePassthrough.assign(intent=RunnableLambda(lambda x: classify_intent.invoke({"query": x["query"]}).intent))
    | RunnableBranch(
        (lambda x: x["intent"] == "full_summary", full_summary_chain),
        retrieval_answer_chain,  # default branch: specific_question
    )
)

# Full pipeline: {url, query} -> answer string
full_pipeline = ingest_chain | RunnablePassthrough.assign(answer=answer_chain)


# --------------------------------------------------------------------------
# 8. Example usage
# --------------------------------------------------------------------------

if __name__ == "__main__":
    result = full_pipeline.invoke({
        "url": "https://www.youtube.com/watch?v=tL9Lw250spc",
        "query": "Summarize this video",
    })
    print(result["answer"])

    # Second call for the same video — ingest_chain will skip re-embedding
    result2 = full_pipeline.invoke({
        "url": "https://www.youtube.com/watch?v=tL9Lw250spc",
        "query": "What happened to Tusko the elephant?",
    })
    print(result2["answer"])

    # Pre-indexing multiple videos concurrently, if you have a batch of URLs:
    # ingest_chain.batch([{"url": u} for u in list_of_urls])